# Advanced IR: Intro to pytorch (Part 1)

The first part of this tutorial is an introduction to pytorch (https://pytorch.org/) a very popular deep learning library for python. We won't be directly using pytorch, but instead a wrapper around pytorch called lightning (https://lightning.ai/). Take a minute now to have a read of these websites to understand their goals.

In this part of the tutorial, you are presented with a lightning class for a binary classifier that will detect positive/negative sentiment in text. The idea is to get you familiar with the lightning deep learning library so that you can better understand ranking models defined in it.

If you do not have access to a computer with a GPU, it is strongly suggested that you run this notebook in Google Colab.

First, we install the libraries

In [1]:
!pip3 install lightning datasets transformers scikit-learn wandb

Then, we import all our libraries

In [2]:
from typing import Any
import os

import lightning as L
import datasets
from datasets import Dataset
import torch
from torch import nn, optim
from torch.nn.functional import sigmoid, binary_cross_entropy
from transformers import AutoModel, AutoConfig, AutoTokenizer
from lightning.pytorch.loggers import WandbLogger
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from torch.autograd import Variable
from torch.utils.data import DataLoader
from tqdm import tqdm

device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
if torch.backends.mps.is_available():
    device = "mps"

/home/kpch/miniconda3/envs/air/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The following class definition contains a definition for the binary classifier. In lightning, we just need to specify the forward pass through the model and the training step, and the library handles everything else for us. In the forward pass, we show create embeddings for each of the words in the sequence, and then use the very first token in the sequence as our representation for it. We then feed this representation into a small linear layer where the input dimensions are the same as our representation (the hidden size) and there is just a single output dimension corresponding to the prediction of the model (0->negative,1->positive sentiment). 

Take some time now to understand the model below.

In [3]:
class SentimentClassifier(L.LightningModule):
    def __init__(self):
        super().__init__()
        model_name = "sentence-transformers/all-MiniLM-L12-v2"
        self.encoder = AutoModel.from_pretrained(model_name)
        self.tokeniser = AutoTokenizer.from_pretrained(model_name)
        config = AutoConfig.from_pretrained(model_name)
        self.classifier = nn.Linear(config.hidden_size, 1)
        self.save_hyperparameters()

    def forward(self, input_ids) -> Any:
        embeddings = self.encoder(input_ids)
        h = embeddings.last_hidden_state[..., 0, :]
        return self.classifier(h)

    def training_step(self, batch):
        input_ids, y = batch["input_ids"], batch["label"]
        y_hat = self.forward(input_ids)
        y_hat = torch.flatten(y_hat)
        y_hat = sigmoid(y_hat)
        loss = binary_cross_entropy(y_hat, y.float())
        self.log("train_loss", loss, logger=True, prog_bar=True, on_step=True, on_epoch=True)
        return loss

    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=1e-5)


If we instantiate an instance of our classifier, we can observe the model architecture that was described above. We have a BERT Transformer encoder model to create a representation of our text sequence, and a linear classification layer to output a score.

In [4]:
classifier = SentimentClassifier()
classifier

SentimentClassifier(
  (encoder): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 384, padding_idx=0)
      (position_embeddings): Embedding(512, 384)
      (token_type_embeddings): Embedding(2, 384)
      (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
              (LayerNorm): LayerNorm((384,), eps=1e-12, el

In the next cell, we download and process the dataset. We perform a pre-processing step called tokenisation which maps words in the sequence into numerical IDs that the model can use.

In [5]:
if not os.path.exists("dataset_imdb"):
    input_dataset = datasets.load_dataset("stanfordnlp/imdb", split="train").to_dict()
    input_ids = classifier.tokeniser(input_dataset["text"], padding="max_length", truncation="only_first")["input_ids"]
    input_dataset["input_ids"] = input_ids
    dataset = Dataset.from_dict(input_dataset)
    dataset.save_to_disk("dataset_imdb")
else:
    dataset = datasets.load_from_disk("dataset_imdb")
dataset.set_format(type="torch", columns=["input_ids", "label"])
dataset.shuffle()

Saving the dataset (1/1 shards): 100%|██████████| 25000/25000 [00:00<00:00, 518032.75 examples/s]


Dataset({
    features: ['text', 'label', 'input_ids'],
    num_rows: 25000
})

With the dataset prepared, we can train the model.

Training the model will likely take about 20-30mins if you run this in Google Colab, so while you train this model, you might want to start on Part 2, or come back to this part later.

In [6]:
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)
trainer = L.Trainer(max_epochs=3, enable_checkpointing=True)
trainer.fit(model=classifier, train_dataloaders=train_loader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/kpch/miniconda3/envs/air/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
/home/kpch/miniconda3/envs/air/lib/python3.12/site-packages/torch/__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.ba

Epoch 0:   0%|          | 0/782 [00:00<?, ?it/s]

We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


Epoch 2: 100%|██████████| 782/782 [01:39<00:00,  7.86it/s, v_num=0, train_loss_step=0.00888, train_loss_epoch=0.0908]

`Trainer.fit` stopped: `max_epochs=3` reached.


Epoch 2: 100%|██████████| 782/782 [01:40<00:00,  7.81it/s, v_num=0, train_loss_step=0.00888, train_loss_epoch=0.0908]


Now we should have our fine-tuned deep learning sentiment classifier! 

Adjust the code below to point to where the trainer output the latest checkpoint.

In [8]:
test_dataset = datasets.load_dataset("stanfordnlp/imdb", split="test")

# wandb will put this into a folder called something like
# /content/bert-sentiment-classifier/dl81te03/checkpoints/epoch=2-step=2346.ckpt
model = SentimentClassifier.load_from_checkpoint("./lightning_logs/version_0/checkpoints/epoch=2-step=2346.ckpt")

test_dataset.set_format(type="torch", columns=["text", "label"])
test_dataset = dataset
test_loader = DataLoader(test_dataset, batch_size=32)

labels = []
with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids, y = batch["input_ids"], batch["label"]
        y_hat = model(input_ids.to(device))
        labels += y_hat.flatten().tolist()

labels = torch.tensor(labels)
labels = sigmoid(labels)
t = Variable(torch.Tensor([0.5]))
labels = (labels > t).float() * 1
print(accuracy_score(dataset["label"], labels.tolist()))

100%|██████████| 782/782 [00:29<00:00, 26.73it/s]


0.98936


You should obtain >90% accuracy.

If you haven't completed Part 2, please go there now.

If you are looking for more excersies to do once you have completed Part 2, consider the following:

1. Modify the sentiment classifier model to support two output dimensions (true/false) instead of just a single score.
2. Get this setup working with a configuration file: https://lightning.ai/docs/pytorch/stable/cli/lightning_cli_advanced.html
3. Train and evaluate the classifier model on another dataset of your choice by looking for an interesting text classification task on huggingface: https://huggingface.co/datasets?task_categories=task_categories:text-classification&sort=trending